In [8]:
import os
from dotenv import load_dotenv
from agents.db import get_connection, query_df
from agents.planner import planner_agent
from agents.checker import checker_agent, get_live_alerts
from agents.retriever import search_policies
from agents.replanner import replanner_agent
from agents.monitor import start_trip, check_trip

load_dotenv()
conn = get_connection()
MBTA_KEY = os.getenv("MBTA_KEY")
print("Ready ✅")

c:\Users\agast\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\snowflake\connector\connection\_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Ready ✅


In [2]:
for t in ["RAW.ROUTES", "RAW.STATIONS", "RAW.ROUTE_STOPS", "RAW.FACILITIES",
          "RAW.ALERTS_HISTORY", "RAG.POLICY_CHUNKS"]:
    n = query_df(conn, f"SELECT COUNT(*) AS N FROM TRANSITWISE_DB.{t}")["N"][0]
    print(f"{t:22} {n:>6} rows")

RAW.ROUTES                 21 rows
RAW.STATIONS              257 rows
RAW.ROUTE_STOPS           337 rows
RAW.FACILITIES            584 rows
RAW.ALERTS_HISTORY      10808 rows
RAG.POLICY_CHUNKS          64 rows


In [3]:
def show(o, d, avoid=None):
    r = planner_agent(conn, o, d, avoid=avoid)
    print(f"{o} → {d}" + (f"  (avoid {avoid})" if avoid else ""))
    if r["ok"]:
        print("  " + " → ".join(r["stations"]))
        print(f"  transfers: {r['transfers'] or 'none'} | ~{r['minutes']} min\n")
    else:
        print("  FAILED: " + r["reason"] + "\n")

show("Harvard", "Back Bay")
show("Harvard", "Back Bay", avoid=["place-dwnxg"])

Harvard → Back Bay
  Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay
  transfers: ['Downtown Crossing'] | ~23 min

Harvard → Back Bay  (avoid ['place-dwnxg'])
  Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Government Center → Haymarket → State → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay
  transfers: ['Park Street', 'Haymarket'] | ~33 min



In [4]:
plan = planner_agent(conn, "Harvard", "Back Bay")
check = checker_agent(plan, MBTA_KEY)
print("Safe:", check["safe"])
for p in check["all_issues"]:
    tag = "BLOCKS" if p in check["blocking"] else "warning"
    print(f"  [{tag}] {p['station']} ({p['role']}): {p['alert'][:80]}")

Safe: False
  [warning] Back Bay (destination): Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.
  [BLOCKS] Downtown Crossing (transfer): Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) is 
  [warning] Chinatown (pass-through): Chinatown Escalator 377 (Oak Grove platform to Washington Street) is unavailable


In [ ]:
for r in search_policies(conn, "Elevator 869 is closed at Downtown Crossing. What should I do?"):
    preview = r["chunk"][:150].rsplit(" ", 1)[0] + " …"
    print(f"{r['score']} | {r['source']}\n   {preview}\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3960.48it/s]


0.904 | mbta_elevator_alerts.pdf | form to paid lobby) is unavailable due to maintenance. Downtown Crossing Downtown Crossing
0.824 | station_downtown_crossing.pdf | Stations › Downtown Crossing Downtown Crossing Red Line Alewife <1 minute away › Ashmont 3
0.689 | mbta_elevator_alerts.pdf | oadway Broadway Escalator Closure ONGOING Broadway Escalator 367 Platform to paid lobby) 
0.612 | mbta_elevator_alerts.pdf | uesday, December 1. ⌄ State Escalator Closure ONGOING State Escalator 6 Oak Grove platfor


In [6]:
result = replanner_agent(conn, "Harvard", "Back Bay", MBTA_KEY)
if result["ok"] and result.get("changed"):
    print("Problems:", [p["station"] for p in result["problems"]])
    print("Detour:", " → ".join(result["new_plan"]["stations"]) if result["new_plan"] else "none found")
    print("\n" + result["explanation"])
    print("\nSources:", result["sources"], "| Guardrails fired:", result["guardrails"])
else:
    print(result.get("explanation") or result.get("reason"))

Problems: ['Downtown Crossing']
Detour: Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Government Center → Haymarket → State → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay

**The problem:** Elevator 869 at Downtown Crossing, which connects the Alewife (Red Line) platform to the Summer Street Concourse, is out of service through Tuesday, December 1 [station_downtown_crossing.pdf]. Your original route transfers at Downtown Crossing, so this closure affects it.

**Recommended option:** Use our planner's alternative route: Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Government Center → Haymarket → State → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay.

**Caveat:** This route still passes through Downtown Crossing, and the alerts don't confirm that it avoids Elevator 869 specifically. Downtown Crossing's station page shows 5 of 6 elevators working [station_downtown_crossing.pdf], but that doesn't identify which one is dow

In [7]:
trip = start_trip(conn, "Harvard", "Back Bay", MBTA_KEY)
trip["known_alerts"] = set()        # simulate outages appearing mid-trip
event = check_trip(conn, trip, MBTA_KEY)
print("New problems detected:", [p["station"] for p in event["new_problems"]])
print("Replanner triggered:", event["replan"] is not None)

New problems detected: ['Downtown Crossing']
Replanner triggered: True
